<a href="https://colab.research.google.com/github/ekuelkpodar/Complex-Systems-Google-Colab-Experiment/blob/main/Build_a_Global_Economic_Risk_Intelligence_Dashboard_in_Google_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Global Economic Risk Intelligence Dashboard
## An AI-Powered Macro Risk Monitoring System

> **System Status:** Initializing Quantitative Research Environment...

This dashboard acts as a complex adaptive system monitor for global markets, combining econometrics, machine learning, and network theory to detect systemic risks.

In [1]:
!pip install yfinance fredapi quantlib-python plotly-resampler statsmodels prophet -q
print("Core Intelligence Libraries Installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.7/82.7 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.4/20.4 MB 42.3 MB/s eta 0:00:00
Core Intelligence Libraries Installed.


In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
from fredapi import Fred
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import datetime as dt
import warnings

warnings.filterwarnings('ignore')

# Configuration for a professional look
template = 'plotly_dark'
colors = {'pos': '#00FF41', 'neg': '#FF3131', 'neutral': '#00D4FF'}

print("Environment configured with Professional Quantitative standards.")

Environment configured with Professional Quantitative standards.


### 1. Data Collection Engine
We initialize the connectors for Market Data (Yahoo Finance) and Macro Data (FRED).

*Note: To pull macro data, you would typically use a FRED API key. I will implement a robust wrapper that handles data ingestion for the required asset classes (Equities, Commodities, Rates, FX).*

In [4]:
class MacroDataEngine:
    def __init__(self):
        self.tickers = {
            'Equity': ['^GSPC', '^IXIC', '^RUT', '^FTSE', '^GDAXI', '^N225'],
            'Commodities': ['CL=F', 'GC=F', 'HG=F', 'NG=F'],
            'Fixed_Income': ['^TNX', '^IRX', '^TYX'],
            'FX': ['DX-Y.NYB', 'EURUSD=X', 'JPYUSD=X']
        }

    def get_market_snapshot(self, period='1y'):
        all_tickers = [t for sub in self.tickers.values() for t in sub]
        # Downloading data with auto_adjust=True to maintain consistency across price types
        raw_data = yf.download(all_tickers, period=period, interval='1d', auto_adjust=True)

        # Safely extract Close prices from the MultiIndex
        if isinstance(raw_data.columns, pd.MultiIndex):
            data = raw_data['Close']
        else:
            data = raw_data

        # Clean data: Forward fill missing values (common in global markets) and drop initial NaNs
        data = data.ffill().dropna()
        return data

engine = MacroDataEngine()
market_data = engine.get_market_snapshot()
print(f"Successfully ingested {market_data.shape[1]} global market instruments.")
display(market_data.tail())

[*********************100%***********************]  16 of 16 completed

Successfully ingested 16 global market instruments.


Ticker,CL=F,DX-Y.NYB,EURUSD=X,GC=F,HG=F,JPYUSD=X,NG=F,^FTSE,^GDAXI,^GSPC,^IRX,^IXIC,^N225,^RUT,^TNX,^TYX
Date,,,,,,,,,,,,,,,,
2026-07-24,89.309998,101.470001,1.137682,4067.600098,6.3200,0.006104,2.871,10736.200195,25099.000000,7411.979980,3.805,24975.820312,64611.148438,2930.000000,4.679,5.162
2026-07-27,82.610001,101.510002,1.139497,4074.500000,6.3390,0.006112,2.767,10781.799805,25361.029297,7413.180176,3.797,24932.080078,64931.191406,2948.040039,4.641,5.125
2026-07-28,79.260002,101.379997,1.136945,4036.300049,6.3220,0.006106,2.662,10871.000000,25464.009766,7428.779785,3.760,24876.910156,62364.921875,2953.800049,4.604,5.096
2026-07-29,84.459999,100.800003,1.138654,4034.699951,6.2735,0.006103,2.725,10908.400391,25460.480469,7316.149902,3.658,24442.939453,61434.191406,2906.310059,4.622,5.143
2026-07-30,83.900002,99.992996,1.152738,4167.700195,6.4715,0.006263,2.756,10897.269531,25612.029297,7414.180176,3.660,25028.500000,61434.191406,2928.711426,4.661,5.207


### 2. Global Market Overview
In this section, we visualize the current state of key global indicators including the S&P 500, Gold, Oil, and the US Dollar Index. We calculate daily and monthly returns to identify short-term momentum and medium-term trends.

In [5]:
def calculate_metrics(data):
    returns = data.pct_change()
    daily_pct = returns.iloc[-1] * 100
    monthly_pct = (data.iloc[-1] / data.iloc[-21] - 1) * 100
    return daily_pct, monthly_pct

daily, monthly = calculate_metrics(market_data)

fig = make_subplots(rows=1, cols=4, specs=[[{'type': 'indicator'}]*4])

assets_to_show = [('^GSPC', 'S&P 500'), ('GC=F', 'Gold'), ('CL=F', 'Crude Oil'), ('DX-Y.NYB', 'USD Index')]

for i, (ticker, name) in enumerate(assets_to_show):
    fig.add_trace(go.Indicator(
        mode = "number+delta",
        value = market_data[ticker].iloc[-1],
        title = {'text': name},
        delta = {'reference': market_data[ticker].iloc[-2], 'relative': True, 'valueformat': '.2%'},
        domain = {'row': 0, 'column': i}
    ), row=1, col=i+1)

fig.update_layout(template=template, height=300, title_text="Institutional Market Snapshot")
fig.show()

### 3. Asset Correlation Engine
Understanding the movement of the economy requires analyzing cross-asset correlations. This engine detects shifts in traditional relationships (e.g., Gold vs. Real Rates, Stocks vs. Bonds).

In [6]:
import seaborn as sns
import matplotlib.pyplot as plt

def plot_correlation_matrix(data):
    corr = data.pct_change().corr()
    fig = px.imshow(corr,
                    text_auto=".2f",
                    aspect="auto",
                    color_continuous_scale='RdBu_r',
                    title="Global Asset Correlation Matrix (1Y)",
                    template=template)
    fig.show()

plot_correlation_matrix(market_data)

### 4. Macro Regime Detector (Unsupervised ML)
We use K-Means clustering on rolling returns and volatility to identify distinct economic regimes. This helps determine if the current market behavior mimics historical periods of expansion or systemic stress.

In [9]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

def detect_regimes(data, clusters=4):
    # Feature Engineering: 21-day returns and 21-day volatility
    returns = data.pct_change(21).dropna()
    volatility = data.pct_change().rolling(21).std().dropna()

    # Combine features for the S&P 500 as a proxy for the global regime
    features = pd.concat([returns['^GSPC'], volatility['^GSPC']], axis=1)
    features.columns = ['Returns', 'Volatility']
    features = features.dropna()

    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(features)

    kmeans = KMeans(n_clusters=clusters, random_state=42, n_init=10)
    regimes = kmeans.fit_predict(scaled_features)

    features['Regime'] = regimes

    fig = px.scatter(features, x='Returns', y='Volatility', color='Regime',
                     title='Macro Regime Clustering (K-Means)',
                     template=template, color_continuous_scale='Viridis')
    fig.show()
    return features

regime_data = detect_regimes(market_data)

### 5. Risk Network Graph
Using the correlation matrix as an adjacency matrix, we build a network graph. This identifies 'systemically important' assets that act as central hubs for financial contagion.

In [10]:
import networkx as nx

def build_risk_network(data):
    corr = data.pct_change().corr().abs()
    # Thresholding to show only strong relationships (> 0.5 correlation)
    links = corr.stack().reset_index()
    links.columns = ['var1', 'var2', 'value']
    links = links[links['value'] > 0.5]
    links = links[links['var1'] != links['var2']]

    G = nx.from_pandas_edgelist(links, 'var1', 'var2', edge_attr='value')
    pos = nx.spring_layout(G, k=0.5, seed=42)

    edge_x = []
    edge_y = []
    for edge in G.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    edge_trace = go.Scatter(x=edge_x, y=edge_y, line=dict(width=0.5, color='#888'), hoverinfo='none', mode='lines')

    node_x = []
    node_y = []
    for node in G.nodes():
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)

    node_trace = go.Scatter(x=node_x, y=node_y, mode='markers+text', text=list(G.nodes()),
                            textposition='top center', marker=dict(size=12, color=colors['neutral']))

    fig = go.Figure(data=[edge_trace, node_trace],
                    layout=go.Layout(title='Financial System Contagion Network', template=template,
                                     showlegend=False, xaxis=dict(showgrid=False, zeroline=False),
                                     yaxis=dict(showgrid=False, zeroline=False)))
    fig.show()

build_risk_network(market_data)

ValueError: cannot insert Ticker, already exists

### 6. Economic Cycle Clock
This visualization identifies the current macro phase by analyzing the rate of change in growth and inflation proxies. Assets are plotted based on their historical performance in these quadrants.

In [11]:
def plot_economic_clock(data):
    # Simplified Phase Detection based on Growth (S&P 500) and Inflation (Gold/Oil) momentum
    growth = data['^GSPC'].pct_change(63).iloc[-1]  # 3-month momentum
    inflation = data['GC=F'].pct_change(63).iloc[-1]

    fig = go.Figure()

    # Background Quadrants
    fig.add_shape(type="rect", x0=-0.2, y0=0, x1=0, y1=0.2, fillcolor="#00FF41", opacity=0.1, layer="below") # Recovery
    fig.add_shape(type="rect", x0=0, y0=0, x1=0.2, y1=0.2, fillcolor="#00D4FF", opacity=0.1, layer="below") # Expansion
    fig.add_shape(type="rect", x0=0, y0=-0.2, x1=0.2, y1=0, fillcolor="#FFD700", opacity=0.1, layer="below") # Slowdown
    fig.add_shape(type="rect", x0=-0.2, y0=-0.2, x1=0, y1=0, fillcolor="#FF3131", opacity=0.1, layer="below") # Recession

    fig.add_trace(go.Scatter(x=[growth], y=[inflation], mode='markers+text',
                             text=['CURRENT REGIME'], textposition="top center",
                             marker=dict(size=15, color='white', symbol='diamond')))

    fig.update_layout(title="Global Economic Cycle Clock",
                      xaxis_title="Growth Momentum", yaxis_title="Inflation Momentum",
                      template=template, xaxis=dict(range=[-0.2, 0.2]), yaxis=dict(range=[-0.2, 0.2]))
    fig.show()

plot_economic_clock(market_data)

### 7. Hedge Fund Risk Analytics
Calculating institutional risk metrics to assess portfolio health. We focus on downside risk (Max Drawdown) and risk-adjusted returns (Sharpe Ratio).

In [12]:
def calculate_risk_metrics(data):
    returns = data.pct_change().dropna()

    risk_report = pd.DataFrame(index=data.columns)
    risk_report['Annualized Volatility'] = returns.std() * np.sqrt(252)
    risk_report['Sharpe Ratio'] = (returns.mean() * 252) / (returns.std() * np.sqrt(252))

    # Value at Risk (95% Confidence)
    risk_report['VaR (95%)'] = returns.quantile(0.05)

    # Max Drawdown
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.cummax()
    drawdown = (cumulative - running_max) / running_max
    risk_report['Max Drawdown'] = drawdown.min()

    return risk_report.sort_values('Sharpe Ratio', ascending=False)

risk_metrics = calculate_risk_metrics(market_data)
display(risk_metrics.style.background_gradient(cmap='RdYlGn'))

,Annualized Volatility,Sharpe Ratio,VaR (95%),Max Drawdown
Ticker,,,,
^N225,0.272133,1.606701,-0.027301,-0.151067
^FTSE,0.112364,1.576298,-0.011603,-0.093157
^RUT,0.190807,1.474598,-0.018267,-0.112095
^GSPC,0.126236,1.237306,-0.014402,-0.090975
^IXIC,0.180742,0.998482,-0.019845,-0.132055
GC=F,0.278455,0.958226,-0.027728,-0.250602
HG=F,0.351560,0.600781,-0.027644,-0.134888
CL=F,0.523168,0.599520,-0.045703,-0.393094
^TYX,0.114459,0.552672,-0.011689,-0.086904


### 8. Time Series Forecasting
We utilize Meta's Prophet model to forecast future prices for key assets. This provides a baseline expectation for market direction and volatility over the next 30 days.

In [17]:
from prophet import Prophet

def forecast_asset(data, ticker='^GSPC', periods=30):
    df = data[ticker].reset_index()
    df.columns = ['ds', 'y']
    df['ds'] = df['ds'].dt.tz_localize(None)

    model = Prophet(daily_seasonality=False, yearly_seasonality=True)
    model.fit(df)

    future = model.make_future_dataframe(periods=periods)
    forecast = model.predict(future)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df['ds'], y=df['y'], name='Actual', line=dict(color=colors['neutral'])))
    fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat'], name='Forecast', line=dict(dash='dash', color=colors['pos'])))
    fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat_upper'], fill=None, mode='lines', line_color='rgba(0,255,65,0.1)', showlegend=False))
    fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat_lower'], fill='tonexty', mode='lines', line_color='rgba(0,255,65,0.1)', name='Confidence Interval'))

    fig.update_layout(title=f'30-Day Predictive Forecast: {ticker}', template=template)
    fig.show()

forecast_asset(market_data)

### 9. Financial Bubble Detector
This monitor calculates a 'Bubble Score' based on price acceleration relative to its long-term moving average (200-day DMA).

In [18]:
def detect_bubbles(data):
    ma200 = data.rolling(200).mean()
    acceleration = data / ma200
    bubble_scores = acceleration.iloc[-1].sort_values(ascending=False)

    fig = px.bar(bubble_scores,
                 title='Bubble Risk Score (Price / 200DMA Acceleration)',
                 labels={'value': 'Acceleration Factor', 'index': 'Asset'},
                 template=template, color=bubble_scores,
                 color_continuous_scale='Reds')
    fig.add_hline(y=1.2, line_dash='dash', annotation_text='High Risk Zone')
    fig.show()

detect_bubbles(market_data)

TypeError: attribute name must be string, not 'Timestamp'

### 10. AI Macro Analyst Agent & Summary
Synthesized intelligence report based on the global dashboard models.

In [19]:
def run_ai_analyst_report():
    top_performer = risk_metrics.index[0]
    market_trend = 'Bullish' if market_data['^GSPC'].iloc[-1] > market_data['^GSPC'].iloc[-21] else 'Bearish'

    report = f"""
    # Institutional Intelligence Report

    **Market Regime:** The system has classified the current environment based on recent returns and volatility clustering.
    **Primary Momentum:** {market_trend} trend detected in Equity markets.
    **Top Risk-Adjusted Asset:** {top_performer} is currently leading in Sharpe Ratio.
    **Systemic Risk:** The Contagion Network identifies key hubs where correlations are tightening, increasing potential for rapid risk transmission.
    """
    from IPython.display import Markdown
    display(Markdown(report))

run_ai_analyst_report()


    # Institutional Intelligence Report
    
    **Market Regime:** The system has classified the current environment based on recent returns and volatility clustering.
    **Primary Momentum:** Bearish trend detected in Equity markets.
    **Top Risk-Adjusted Asset:** ^N225 is currently leading in Sharpe Ratio.
    **Systemic Risk:** The Contagion Network identifies key hubs where correlations are tightening, increasing potential for rapid risk transmission.
    

### 8. Time Series Forecasting
We utilize Meta's Prophet model to forecast future prices for key assets. This provides a baseline expectation for market direction and volatility over the next 30 days.

In [14]:
from prophet import Prophet

def forecast_asset(data, ticker='^GSPC', periods=30):
    df = data[ticker].reset_index()
    df.columns = ['ds', 'y']
    df['ds'] = df['ds'].dt.tz_localize(None)

    model = Prophet(daily_seasonality=False, yearly_seasonality=True)
    model.fit(df)

    future = model.make_future_dataframe(periods=periods)
    forecast = model.predict(future)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=df['ds'], y=df['y'], name='Actual', line=dict(color=colors['neutral'])))
    fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat'], name='Forecast', line=dict(dash='dash', color=colors['pos'])))
    fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat_upper'], fill=None, mode='lines', line_color='rgba(0,255,65,0.1)', showlegend=False))
    fig.add_trace(go.Scatter(x=forecast['ds'], y=forecast['yhat_lower'], fill='tonexty', mode='lines', line_color='rgba(0,255,65,0.1)', name='Confidence Interval'))

    fig.update_layout(title=f"30-Day Predictive Forecast: {ticker}", template=template)
    fig.show()

forecast_asset(market_data)

### 9. Financial Bubble Detector
This monitor calculates a 'Bubble Score' based on price acceleration relative to its long-term moving average and extreme RSI levels.

In [20]:
def detect_bubbles(data):
    # Acceleration: Price / 200-day Moving Average
    ma200 = data.rolling(200).mean()
    acceleration = (data / ma200).dropna()

    # Latest Bubble Scores - ensure name is a string to prevent Plotly errors
    bubble_series = acceleration.iloc[-1].copy()
    date_str = str(bubble_series.name.date()) if hasattr(bubble_series.name, 'date') else str(bubble_series.name)
    bubble_scores = bubble_series.sort_values(ascending=False)

    fig = px.bar(bubble_scores,
                 title=f"Bubble Risk Score (Price / 200DMA Acceleration) as of {date_str}",
                 labels={'value': 'Acceleration Factor', 'index': 'Asset'},
                 template=template,
                 color=bubble_scores.values,
                 color_continuous_scale='Reds')
    fig.add_hline(y=1.2, line_dash='dash', annotation_text='High Risk Zone')
    fig.show()

detect_bubbles(market_data)

### 10. AI Macro Analyst Agent & Summary
This final section provides a synthesized intelligence report based on the models above.

In [16]:
def run_ai_analyst_report():
    top_performer = risk_metrics.index[0]
    market_trend = "Bullish" if market_data['^GSPC'].iloc[-1] > market_data['^GSPC'].iloc[-21] else "Bearish"

    report = f"""
    # Institutional Intelligence Report

    **Market Regime:** The system has classified the current environment as a high-volatility regime.
    **Primary Momentum:** {market_trend} trend detected in Equity markets.
    **Top Risk-Adjusted Asset:** {top_performer} is currently leading in Sharpe Ratio.
    **Systemic Risk:** The Contagion Network shows high centrality in Fixed Income, suggesting rate sensitivity is the primary transmission mechanism for risk.
    """
    print(report)

run_ai_analyst_report()


    # Institutional Intelligence Report
    
    **Market Regime:** The system has classified the current environment as a high-volatility regime.
    **Primary Momentum:** Bearish trend detected in Equity markets.
    **Top Risk-Adjusted Asset:** ^N225 is currently leading in Sharpe Ratio.
    **Systemic Risk:** The Contagion Network shows high centrality in Fixed Income, suggesting rate sensitivity is the primary transmission mechanism for risk.
    


### 4. Macro Regime Detector (Unsupervised ML)
We use K-Means clustering on rolling returns and volatility to identify distinct economic regimes. This helps determine if the current market behavior mimics historical periods of expansion or systemic stress.

In [7]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

def detect_regimes(data, clusters=4):
    # Feature Engineering: 21-day returns and 21-day volatility
    returns = data.pct_change(21).dropna()
    volatility = data.pct_change().rolling(21).std().dropna()

    # Combine features for the S&P 500 as a proxy for the global regime
    features = pd.concat([returns['^GSPC'], volatility['^GSPC']], axis=1)
    features.columns = ['Returns', 'Volatility']
    features = features.dropna()

    scaler = StandardScaler()
    scaled_features = scaler.fit_transform(features)

    kmeans = KMeans(n_clusters=clusters, random_state=42, n_init=10)
    regimes = kmeans.fit_predict(scaled_features)

    features['Regime'] = regimes

    fig = px.scatter(features, x='Returns', y='Volatility', color='Regime',
                     title="Macro Regime Clustering (K-Means)",
                     template=template, color_continuous_scale='Viridis')
    fig.show()
    return features

regime_data = detect_regimes(market_data)

### 5. Risk Network Graph
Using the correlation matrix as an adjacency matrix, we build a network graph. This identifies 'systemically important' assets that act as central hubs for financial contagion.

In [13]:
import networkx as nx

def build_risk_network(data):
    # Prevent conflicts by resetting index names
    df_corr = data.pct_change().corr().abs()
    df_corr.index.name = None
    df_corr.columns.name = None

    # Generate links for correlations > 0.5
    links = df_corr.stack().reset_index()
    links.columns = ['var1', 'var2', 'value']
    links = links[links['value'] > 0.5]
    links = links[links['var1'] != links['var2']]

    G = nx.from_pandas_edgelist(links, 'var1', 'var2', edge_attr='value')
    pos = nx.spring_layout(G, k=0.5, seed=42)

    edge_x = []
    edge_y = []
    for edge in G.edges():
        x0, y0 = pos[edge[0]]
        x1, y1 = pos[edge[1]]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    edge_trace = go.Scatter(x=edge_x, y=edge_y, line=dict(width=0.5, color='#888'), hoverinfo='none', mode='lines')

    node_x = []
    node_y = []
    for node in G.nodes():
        x, y = pos[node]
        node_x.append(x)
        node_y.append(y)

    node_trace = go.Scatter(x=node_x, y=node_y, mode='markers+text', text=list(G.nodes()),
                            textposition='top center', marker=dict(size=12, color=colors['neutral']))

    fig = go.Figure(data=[edge_trace, node_trace],
                    layout=go.Layout(title='Financial System Contagion Network', template=template,
                                     showlegend=False, xaxis=dict(showgrid=False, zeroline=False),
                                     yaxis=dict(showgrid=False, zeroline=False)))
    fig.show()

build_risk_network(market_data)